# 業務PC GPU学習ワークフロー

画像準備 → 検査 → smoke test → 本学習 → 評価を、CLIと同じ実装で実行します。先に `docs/business-pc-workflow.md` を読み、VS Codeで `.venv` kernelを選択してください。業務データはリポジトリ外に置きます。

In [ ]:
%pip install -e ".[train,dev]"

## 1. Python・GPU診断
`cuda_available: True`とRTXの名前が表示されることを確認します。

In [ ]:
import subprocess
import sys
from pathlib import Path


def run(*arguments: str) -> None:
    subprocess.run([sys.executable, "-m", "fm_to_edge_seg", *arguments], check=True)


run("doctor")

## 2. 業務PC内のパス設定
自分のPCに合わせて3か所を変更します。metadataを使わない場合は `METADATA = None` にします。

In [ ]:
SOURCE = Path(r"D:\fm-to-edge-seg-data\source")
PREPARED = Path(r"D:\fm-to-edge-seg-data\prepared")
METADATA = SOURCE / "metadata.csv"
CONFIG = Path("configs/experiment/local_business_baseline.yaml")
OUTPUT = Path("artifacts/business_baseline_v001")

## 3. データ変換・検査・目視確認
初回だけ変換セルを実行します。再生成時は末尾へ `--overwrite` 相当を追加する前に、変換先が正しいか確認してください。

In [ ]:
arguments = ["prepare-binary-dataset", str(SOURCE), str(PREPARED), "--val-fraction", "0.2"]
if METADATA is not None:
    arguments += ["--metadata", str(METADATA)]
run(*arguments)

In [ ]:
MANIFEST = PREPARED / "manifest.csv"
run("validate-manifest", str(MANIFEST))
run(
    "preview-dataset",
    str(MANIFEST),
    "artifacts/business_data_preview.png",
    "--split",
    "train",
    "--limit",
    "12",
)

`artifacts/business_data_preview.png`をVS Codeで開き、赤いmaskが糸と一致することを確認します。続いてexample configをコピーし、manifestパスを編集してください。

In [ ]:
# Smoke test: 精度ではなく、GPU上で最後まで動くことの確認
run(
    "train",
    str(CONFIG),
    "--epochs",
    "1",
    "--num-workers",
    "0",
    "--max-train-batches",
    "2",
    "--max-validation-batches",
    "1",
)

In [ ]:
# Full training: 出力先をsmoke testと分けてから実行することを推奨
run("train", str(CONFIG))

## 4. 評価
test splitがなければ `SPLIT = 'val'` にします。

In [ ]:
SPLIT = "val"
run(
    "evaluate",
    str(CONFIG),
    str(OUTPUT / "best.pt"),
    str(OUTPUT / f"{SPLIT}_evaluation"),
    "--split",
    SPLIT,
)